## 1. 인증 설정
`.env`에서 NCP API 키를 불러와 지역검색 API 호출에 쓸 `url`, `headers`를 만듦.

In [1]:
import os
import dotenv
import requests
import pandas as pd

dotenv.load_dotenv()

client_id = os.getenv("client_ID")
client_secret = os.getenv("client_secret")

# curl.exe --location -G "https://naverapihub.apigw.ntruss.com/search/v1/local" --data-urlencode "query=정자동 카페" -d "display=2" -d "start=1" -d "sort=comment" -d "format=json" --header "X-NCP-APIGW-API-KEY-ID: {}" --header "X-NCP-APIGW-API-KEY: {}"

url = "https://naverapihub.apigw.ntruss.com/search/v1/local"
headers = {
    "X-NCP-APIGW-API-KEY-ID": client_id,
    "X-NCP-APIGW-API-KEY": client_secret,
}

## 2. 마포구 놀거리/생활업종 대량 수집
13개 동 × 카테고리(놀거리, 카페/맛집/술집)를 순회 호출 → 기존 CSV와 병합 후 `title`+`address` 기준 중복 제거해서 저장. 재실행해도 안전(누적 방식).

In [2]:
import os
import time

from nolda_common import MAPO_DONGS

CSV_PATH = "data/naver_local_search_mapo.csv"

# 마포구 행정동 x 업종 조합으로 순회 호출 -> 결과 누적
dongs = MAPO_DONGS
categories = [
    # 놀거리 (우선순위)
    "오락실", "볼링장", "방탈출", "보드게임카페", "당구장",
    "만화카페", "VR체험", "키즈카페", "스크린야구", "노래방",
    "PC방", "찜질방", "클라이밍", "공원",
    # 전시/소품샵/야경명소
    "전시", "소품샵", "루프탑", "전망대", "야경",
    # 생활 업종 (필요한 것만)
    "카페", "술집",
    # 음식점 - "맛집" 하나로는 동당 display=5 cap에 걸려 다양성이 부족해서 세분화
    "맛집", "한식", "일식", "중식", "양식", "분식", "브런치", "고깃집",
    # 문구점 - '말랑이' 등 트렌드 상품은 업체 단위 API로는 필터링이 안 되므로,
    # 일단 문구점 목록을 모아두고 실제 취급 여부는 별도 확인 필요
    # "문구점",
]

rows = []
for dong in dongs:
    for category in categories:
        query = f"마포구 {dong} {category}"
        params = {
            "query": query,
            "display": 5,
            "start": 1,
            "sort": "comment",
            "format": "json",
        }
        resp = requests.get(url, headers=headers, params=params)
        if resp.status_code != 200:
            print(f"실패: {query} ({resp.status_code})")
            continue

        items = resp.json().get("items", [])
        for item in items:
            item["dong"] = dong
            item["category_keyword"] = category
            rows.append(item)

        time.sleep(0.2)  # API 호출 제한 방지

new_df = pd.DataFrame(rows)

# 기존에 저장해둔 데이터(예: 병원/약국 등 이제는 더 안 모으는 업종)는 그대로 유지하고 새 결과만 합침
if os.path.exists(CSV_PATH):
    existing_df = pd.read_csv(CSV_PATH)
    df_all = pd.concat([existing_df, new_df], ignore_index=True)
else:
    df_all = new_df

df_all = df_all.drop_duplicates(subset=["title", "address"]).reset_index(drop=True)
df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

print(f"총 {len(df_all)}건 저장")
df_all

총 921건 저장


,title,link,category,description,telephone,address,roadAddress,mapx,mapy,dong,category_keyword
0,슈가코인노래연습장 공덕점,http://m.facebook.com/sugarcoin1,오락시설>노래방,NaN,NaN,서울특별시 마포구 공덕동 105-159 지하 1층,서울특별시 마포구 마포대로 180 지하 1층,1269555073,375501823,공덕동,오락실
1,뮤즈코인노래연습장,NaN,오락시설>노래방,NaN,NaN,서울특별시 마포구 신공덕동 20-18 지하1층,서울특별시 마포구 백범로37길 22 지하1층,1269548139,375440661,공덕동,오락실
2,클럽스트라이크 <b>볼링장</b>,https://blog.naver.com/strikepub,"스포츠,오락>볼링장",NaN,NaN,서울특별시 마포구 도화동 36 4층 클럽스트라이크 볼링장,서울특별시 마포구 마포대로 52 4층 클럽스트라이크 볼링장,1269476654,375403280,공덕동,볼링장
3,뉴청룡<b>볼링장</b>,NaN,"스포츠,오락>볼링장",NaN,NaN,서울특별시 용산구 갈월동 98-38 청룡빌딩 지하1층,서울특별시 용산구 한강대로 257 청룡빌딩 지하1층,1269726390,375413560,공덕동,볼링장
4,타겟볼링,NaN,"스포츠,오락>볼링장",NaN,NaN,서울특별시 서대문구 대현동 101-7 혜우빌딩 지하1층,서울특별시 서대문구 신촌역로 10 혜우빌딩 지하1층,1269430720,375575890,공덕동,볼링장
...,...,...,...,...,...,...,...,...,...,...,...
916,탭샵바 상암MBC점,https://www.instagram.com/tap.shop.bar/,술집>와인,NaN,NaN,서울특별시 마포구 상암동 1604 mbc몰 플라자 1층 (올리브영 옆),서울특별시 마포구 성암로 255 mbc몰 플라자 1층 (올리브영 옆),1268914645,375797003,상암동,브런치
917,프롬커피바 상암점,https://fromcoffeebar.com/index.html,"음식점>카페,디저트",NaN,NaN,서울특별시 마포구 상암동 1653 이안오피스텔 2단지 1층 프롬커피바 상암점,서울특별시 마포구 월드컵북로 361 이안오피스텔 2단지 1층 프롬커피바 상암점,1268910302,375770913,상암동,브런치
918,버거리 상암DMC점,http://burgerry.co.kr/,음식점>양식>햄버거,NaN,NaN,서울특별시 마포구 상암동 1597 사보이시티디엠씨 212호,서울특별시 마포구 월드컵북로54길 17 사보이시티디엠씨 212호,1268889619,375812993,상암동,브런치
919,축제나우,,"지원,대행>전시,행사대행",,,서울특별시 마포구 대흥동 774 대흥빌딩 2층,서울특별시 마포구 큰우물로 16 대흥빌딩 2층,1269410427,375444376,대흥동,전시


## 3. 카페/맛집 보강
지역검색 API는 쿼리 하나당 결과가 `display`와 무관하게 무조건 5건으로 캡되어 있어서(`total` 필드 확인함),
같은 쿼리를 `sort=comment`/`sort=random`으로 각각 호출하면 서로 다른 업체가 나오는 걸 이용해 커버리지를 늘림.
카페는 세부 키워드(브런치카페/베이커리카페/스터디카페/애견동반카페)도 추가.

In [ ]:
cafe_food_categories = [
    "카페", "브런치카페", "베이커리카페", "애견동반카페",
    "맛집", "한식", "일식", "중식", "양식", "분식", "브런치", "고깃집",
]
sorts = ["comment", "random"]

rows = []
for dong in dongs:
    for category in cafe_food_categories:
        for sort in sorts:
            query = f"마포구 {dong} {category}"
            params = {
                "query": query,
                "display": 5,
                "start": 1,
                "sort": sort,
                "format": "json",
            }
            resp = requests.get(url, headers=headers, params=params)
            if resp.status_code != 200:
                print(f"실패: {query} ({sort}) ({resp.status_code})")
                continue

            items = resp.json().get("items", [])
            for item in items:
                item["dong"] = dong
                item["category_keyword"] = category
                rows.append(item)

            time.sleep(0.2)  # API 호출 제한 방지

new_df = pd.DataFrame(rows)

existing_df = pd.read_csv(CSV_PATH)
df_all = pd.concat([existing_df, new_df], ignore_index=True)
df_all = df_all.drop_duplicates(subset=["title", "address"]).reset_index(drop=True)
df_all.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")

food_kw = {"맛집", "한식", "일식", "중식", "양식", "분식", "브런치", "고깃집"}
cafe_kw = {"카페", "브런치카페", "베이커리카페", "애견동반카페"}
print(f"총 {len(df_all)}건 저장")
print(f"카페류: {df_all['category_keyword'].isin(cafe_kw).sum()}건 / 맛집류: {df_all['category_keyword'].isin(food_kw).sum()}건")